In [38]:
import os
import json
import numpy as np
import random
import time
import re
import tiktoken
from google import genai
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from openai import OpenAI

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# # --- Hugging Face mirror/cache settings (set before loading embedding models) ---
# os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
# os.environ.setdefault("HUGGINGFACE_HUB_CACHE", os.path.expanduser("~/.cache/huggingface"))

# print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT"))
# print("HUGGINGFACE_HUB_CACHE:", os.environ.get("HUGGINGFACE_HUB_CACHE"))

In [91]:

# --- Configuration ---
EMBEDDING_DIM = 384  # Dimension for question and answer embeddings

UPDATE_FREQUENCY = 1    # Update JSON records after every question


# TODO: configure more parameters here
# Regularization parameter λ, exploration coefficient γ, step size η, network width m, 
# gradient descent steps J, LLM pool size M , batch size B_batch

LEARNING_RATE = 0.01 # 学习率
REG = 2 #正则化参数
M = 7 # LLM pool size
GEMMA = 1 # 探索系数
L = 3 # 2层隐藏层
BATCH_SIZE = 5 # B_batch
WIDTH = 100 # m
GD_STEPS = 10 # J

In [40]:
# --- CUDA / Device Setup ---
# 自动选择 GPU；若不可用则回退到 CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")
else:
    print("Using CPU")


def to_device(x):
    """Move tensor/module to selected device."""
    if hasattr(x, "to"):
        return x.to(DEVICE)
    return x


# 可选：提高矩阵乘法性能（Ampere 及以上通常有效）
torch.set_float32_matmul_precision("high")

Torch version: 2.10.0+cu128
CUDA available: True
CUDA runtime: 12.8
GPU count: 2
Current GPU: Quadro RTX 8000


In [41]:
if DEVICE.type == "cuda":
    # 清理缓存（可选）
    torch.cuda.empty_cache()
    print("CUDA is ready for training/inference.")

CUDA is ready for training/inference.


In [42]:
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    # Notebook 常在 Algorithms/ 目录运行，.env 在项目根目录
    candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
    for p in candidates:
        if p.exists():
            dotenv_path = str(p)
            break
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print("Loaded .env:", dotenv_path)
else:
    print("No .env found; please create one at project root.")


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL")

Loaded .env: /home/guisy/MyExperiment/LLM_DAG_Routing_02/.env


In [ ]:
# --- Test two models via OpenRouter ---
Decompose_MODEL_NAME = "deepseek/deepseek-v3.2"
GRADER_MODEL_NAME = "deepseek/deepseek-v3.2"

assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY 未设置，请检查 .env"
base_url = OPENROUTER_BASE_URL 

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=base_url)

models_to_test = [Decompose_MODEL_NAME, GRADER_MODEL_NAME]
prompt = "请只回复：OK"

for model_name in models_to_test:
    try:
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=16,
            temperature=0.0,
        )
        text = resp.choices[0].message.content if resp.choices else "<no choice>"
        print(f"✅ {model_name} -> {text}")
    except Exception as e:
        print(f"❌ {model_name} failed: {type(e).__name__}: {e}")

❌ google/gemini-2.5-flash-lite failed: PermissionDeniedError: Error code: 403 - {'error': {'message': 'This model is not available in your region.', 'code': 403}}
✅ deepseek/deepseek-v3.2 -> OK


In [44]:
# 从项目根目录的 model_config.py 加载模型列表
import sys
from pathlib import Path

# 当前 notebook 在 Algorithm/ 下，根目录是其上一级
project_root = Path.cwd() if (Path.cwd() / "model_config.py").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from model_config import MODELS_CONFIG

AVAILABLE_LLMS = list(MODELS_CONFIG.keys())
LLM_ID_DIM = len(AVAILABLE_LLMS)

print(f"Loaded {LLM_ID_DIM} models from model_config.py")
print(AVAILABLE_LLMS[:7])

Loaded 7 models from model_config.py
['qwen/qwen-2.5-7b-instruct', 'meta-llama/llama-3.1-8b-instruct', 'mistralai/mistral-small-3.2-24b-instruct', 'google/gemma-3-27b-it', 'meta-llama/llama-3.3-70b-instruct', 'qwen/qwen3.5-flash-02-23', 'mistralai/mistral-nemo']


In [45]:
# TODO: load train set data/final_evaluation_dataset.json
# TODO: 创建记录模型回答的json文件，记录格式为：
# {
#   "problem_id": {
#       "question": "原问题文本",
#       "answers": "原问题答案",
#       "final_status": "success/failure",
#       "steps_taken": 3, #几个节点
#       "attempts": [
#        {
#         "step": ,
#         "task_desc": "",
#         "chosen_model": "",
#         "reward": ,
#         "output": "",
#         "llm_input_messages": [
#           {
#             "role": "system",
#             "content": ""
#           },
#           {
#             "role": "user",
#             "content": ""
#           }
#         ]
#        },
#        { 
#         "step": "2",
#         "task_desc": "",
#         "chosen_model": "",
#         "reward": ,
#         "output": "",
#         "llm_input_messages": [
#           {
#             "role": "system",
#             "content": ""
#           },
#           {
#             "role": "user",
#             "content": ""
#           }
#         ]
#        }
#       ]
#
#   }
# }

# TODO: 保存模型参数文件
# TODO: complete .gitignore to exclude model parameters

In [62]:
import json
import os
import numpy as np
from pathlib import Path

# ==========================================
# TODO: load train set data/final_evaluation_dataset.json
# ==========================================
def load_dataset(file_path="../data/final_evaluation_dataset_500.json"):
    candidates = [
        Path(file_path),
        Path.cwd() / "final_evaluation_dataset_500.json",
        Path.cwd().parent / "final_evaluation_dataset_500.json",
        Path.cwd() / "data" / "final_evaluation_dataset_500.json",
        Path.cwd().parent / "data" / "final_evaluation_dataset_500.json",
    ]
    real_path = next((p for p in candidates if p.exists()), None)
    if real_path is None:
        print(f"⚠️ 警告: 数据集文件不存在。已尝试: {[str(p) for p in candidates]}")
        return []
    with open(real_path, 'r', encoding='utf-8') as f:
        print(f"✅ 成功加载数据集: {real_path}")
        return json.load(f)


def _to_jsonable(obj):
    """Recursively convert numpy / non-JSON-native objects to JSON-safe types."""
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

# ==========================================
# TODO: 创建记录模型回答的json文件
# ==========================================
class ResultRecorder:
    def __init__(self, output_file="execution_records.json"):
        # 仅使用“已有”的 record 目录（优先当前目录，其次上一级目录）
        candidates = [Path.cwd() / "record", Path.cwd().parent / "record"]
        record_dir = next((p for p in candidates if p.exists() and p.is_dir()), None)
        if record_dir is None:
            raise FileNotFoundError("未找到已有的 record 目录，请先创建 record 文件夹。")

        self.output_file = str(record_dir / Path(output_file).name)
        self.records = {}

        # 若记录文件已存在则覆盖；不存在则新建空文件
        with open(self.output_file, 'w', encoding='utf-8') as f:
            json.dump(self.records, f, ensure_ascii=False, indent=2)

    def add_record(self, problem_id: str, question: str, expected_answers: str, 
                   final_status: str, attempts: list):
        """
        按照规定的字典格式写入单条测试记录。
        attempts 应该是一个包含 step, task_desc, chosen_model, reward, output, llm_input_messages 的列表字典。
        """
        self.records[problem_id] = {
            "question": question,
            "answers": expected_answers,
            "final_status": final_status,
            "steps_taken": len(attempts),
            "attempts": _to_jsonable(attempts)
        }
        self.save()

    def save(self):
        """将记录持久化到 JSON 文件中，支持 UPDATE_FREQUENCY"""
        with open(self.output_file, 'w', encoding='utf-8') as f:
            json.dump(_to_jsonable(self.records), f, ensure_ascii=False, indent=2)

In [48]:
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
# TODO: choose an embedding model that can process long contexts.

In [49]:
# TODO
# 将上下文拼接好后再转换为384维向量
class FeatureExtractor:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.model.to(device)
        print("Embedding model loaded successfully.")

    def get_feature_vector(self, text_context):
        # 如果输入为空，返回全0向量（防止报错）
        if not text_context or text_context.strip() == "":
            return np.zeros(EMBEDDING_DIM, dtype=np.float32)
            
        vector = self.model.encode(
            text_context, 
            convert_to_numpy=True, 
            normalize_embeddings=True
        )
        return vector.astype(np.float32)

# 实例化特征提取器
extractor = FeatureExtractor()

Loading embedding model: BAAI/bge-small-en-v1.5...


Loading embedding model: BAAI/bge-small-en-v1.5...


'[Errno 104] Connection reset by peer' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
No sentence-transformers model found with name BAAI/bge-small-en-v1.5. Creating a new one with mean pooling.
'[Errno 104] Connection reset by peer' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


Loading embedding model: BAAI/bge-small-en-v1.5...


'[Errno 104] Connection reset by peer' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
No sentence-transformers model found with name BAAI/bge-small-en-v1.5. Creating a new one with mean pooling.
'[Errno 104] Connection reset by peer' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [ ]:
# TODO: ReLU需要改吗

In [53]:
class LLMNet(nn.Module):
    def __init__(self, input_dim=EMBEDDING_DIM, width=WIDTH, L=L):
        super().__init__()
        num_hidden = max(L - 1, 1)
        layers = []
        in_dim = input_dim
        for _ in range(num_hidden):
            layers.append(nn.Linear(in_dim, width))
            layers.append(nn.ReLU())
            in_dim = width
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        if not torch.is_tensor(x):
            x = torch.tensor(x, dtype=torch.float32)
        if x.dim() == 1:
            x = x.unsqueeze(0)
        return self.net(x).squeeze(-1)


In [82]:
class DAGNode:
    def __init__(self, node_id, task_description):
        self.node_id = node_id
        self.task_description = task_description
        self.predecessors = []
        self.successors = []
        self.layer = 0
        
        # 运行时状态
        self.input_context = ""
        self.output_content = ""      # 模型原始输出（可含思考过程）
        self.answer_content = ""      # 仅保留 <answer>...</answer> 提取结果
        self.selected_model = None
        self.feature_vector = None 
        self.reward = 0.0          

    def add_successor(self, node):
        if node not in self.successors:
            self.successors.append(node)
        if self not in node.predecessors:
            node.predecessors.append(self)

class DAGGraph:
    def __init__(self):
        self.nodes = {}

    def add_node(self, node_id, description):
        if node_id not in self.nodes:
            self.nodes[node_id] = DAGNode(node_id, description)
        return self.nodes[node_id]

    def add_edge(self, source_id, target_id):
        if source_id in self.nodes and target_id in self.nodes:
            self.nodes[source_id].add_successor(self.nodes[target_id])

    def compute_layers(self):
        #层级计算：Layer = max(predecessor_layers) + 1
        for node in self.nodes.values(): node.layer = 0
        for _ in range(len(self.nodes) + 1):
            for node in self.nodes.values():
                for pred in node.predecessors:
                    if node.layer < pred.layer + 1:
                        node.layer = pred.layer + 1
        layers = {}
        for node in self.nodes.values():
            if node.layer not in layers: layers[node.layer] = []
            layers[node.layer].append(node)
        return layers


In [92]:
import json
import re


def decompose_query_to_dag(query_text: str, problem_id: str, client) -> DAGGraph:
    """
    Algorithm 1 Line 4: Decompose q_t into a DAG G_t
    Calls the LLM to decompose the original natural language query into a standardized DAG JSON format.

    If decomposition fails:
    - print error info
    - fallback to a single node (directly answer original question)
    """

    system_prompt = """
    You are an expert in logical task decomposition. Please decompose the user's complex problem into a Directed Acyclic Graph (DAG) of derivation steps.

    CRITICAL RULES:
    1. You are a planner, NOT a solver: Do NOT directly compute answers, simplify equations, or solve the problem in the task descriptions.
    2. Context preservation: Ensure each sub-task description explicitly includes all information it needs from the original problem statement.
    3. Dependency sufficiency: When assigning predecessors for a node, ensure those predecessors can provide ALL required information, physical quantities, and mathematical variables for the current sub-task.
    4. Multiple-choice questions: If the original problem has options, include full options in the final selection node.
    5. Execution output rule: In each node description, require the solver to wrap the final concise answer with <answer>...</answer>.
    6. If the Original Question includes explicit multiple-choice options (e.g., A, B, C, D), create a final node to match the extracted answer to the correct option letter. If the Original Question is open-ended and does NOT contain options, DO NOT hallucinate options. 
    The final node MUST instruct the model to output the exact entity, number, or option required by the Original Question.

    Output MUST be valid JSON only, with this structure (dependencies are written per node, NOT via edges):
    {
      "nodes": [
        {
          "id": "1",
          "description": "Extract known values from the problem and write them clearly. Put final concise result in <answer>...</answer>.",
          "predecessors": []
        },
        {
          "id": "2",
          "description": "Using values from Node 1, apply the required formula to compute the target quantity. Put final concise result in <answer>...</answer>.",
          "predecessors": ["1"]
        },
        {
          "id": "3",
          "description": "Match the computed result from Node 2 to the full options list (A..., B..., ...). Output ONLY the option letter in <answer>...</answer>.",
          "predecessors": ["2"]
        }
      ]
    }
    """

    def _normalize_task_desc(desc: str) -> str:
        return (desc or "").strip()

    def _fallback_graph(err_msg: str) -> DAGGraph:
        print(f"❌ Decomposition failed: {err_msg}")
        g = DAGGraph()
        g.problem_description = query_text
        g.add_node(
            "fallback_node",
            _normalize_task_desc(
                "Answer the following question directly and wrap final concise answer in <answer>...</answer>: " + query_text
            ),
        )
        return g

    user_prompt = f"Problem ID: {problem_id}\nOriginal Question: {query_text}"

    # 1) Call decomposition model and parse JSON
    try:
        resp = client.chat.completions.create(
            model=Decompose_MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=1400,
            temperature=0.2,
        )
        raw_output = (resp.choices[0].message.content or "").strip()

        json_str = raw_output
        if "```json" in json_str:
            json_str = json_str.split("```json", 1)[1].split("```", 1)[0].strip()
        elif "```" in json_str:
            json_str = json_str.split("```", 1)[1].split("```", 1)[0].strip()

        dag_dict = json.loads(json_str)
    except Exception as e:
        return _fallback_graph(f"model call / JSON parse error: {type(e).__name__}: {e}")

    # 2) Build graph (兼容新旧 schema)
    try:
        graph = DAGGraph()
        graph.problem_description = dag_dict.get("problem_description", query_text)

        nodes_data = dag_dict.get("nodes", [])
        if not isinstance(nodes_data, list) or len(nodes_data) == 0:
            raise ValueError("'nodes' missing or empty")

        # 新 schema: id/description/predecessors
        schema_new = all(
            isinstance(n, dict) and ("id" in n) and ("description" in n) and ("predecessors" in n)
            for n in nodes_data
        )
        # 兼容 schema A: node_id/sub_task_description/dependency_node_id
        schema_a = all(isinstance(n, dict) and ("node_id" in n) and ("sub_task_description" in n) for n in nodes_data)
        # 兼容 schema B: id/description + edges
        schema_b = all(isinstance(n, dict) and ("id" in n) and ("description" in n) for n in nodes_data)

        if schema_new:
            for n_data in nodes_data:
                node_id = str(n_data["id"])
                task_desc = _normalize_task_desc(str(n_data["description"]))
                graph.add_node(node_id, task_desc)

            for n_data in nodes_data:
                target_id = str(n_data["id"])
                preds = n_data.get("predecessors", [])
                if not isinstance(preds, list):
                    preds = []
                for source_id in preds:
                    graph.add_edge(str(source_id), target_id)

        elif schema_a:
            for n_data in nodes_data:
                node_id = str(n_data["node_id"])
                task_desc = _normalize_task_desc(str(n_data["sub_task_description"]))
                graph.add_node(node_id, task_desc)
            for n_data in nodes_data:
                target_id = str(n_data["node_id"])
                for source_id in n_data.get("dependency_node_id", []):
                    graph.add_edge(str(source_id), target_id)

        elif schema_b:
            for n_data in nodes_data:
                node_id = str(n_data["id"])
                task_desc = _normalize_task_desc(str(n_data["description"]))
                graph.add_node(node_id, task_desc)

            edges_data = dag_dict.get("edges", [])
            if not isinstance(edges_data, list):
                edges_data = []
            for e in edges_data:
                if isinstance(e, (list, tuple)) and len(e) == 2:
                    source_id, target_id = e
                    graph.add_edge(str(source_id), str(target_id))

        else:
            raise ValueError("unsupported node schema")

        if len(graph.nodes) == 0:
            raise ValueError("no valid nodes after parsing")

        print(f"✅ Successfully decomposed query into a DAG with {len(graph.nodes)} nodes.")
        return graph

    except Exception as e:
        return _fallback_graph(f"graph build error: {type(e).__name__}: {e}")

In [89]:
def _extract_answer_tag(text: str) -> str:
    """Extract concise answer from <answer>...</answer>. If missing, fallback to stripped text."""
    if text is None:
        return ""
    s = str(text)
    matches = re.findall(r"<answer>\s*(.*?)\s*</answer>", s, flags=re.IGNORECASE | re.DOTALL)
    if matches:
        return matches[-1].strip()
    return s.strip()


def _normalize_answer_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    # Keep only alnum + CJK for robust matching
    s = re.sub(r"[^\w\u4e00-\u9fff]", "", s)
    return s


def score_final_answer_by_gt(client, ground_truth_answer: str, final_output: str) -> int:
    """Scorer-1: use GRADER_MODEL_NAME to judge whether final answer is correct (0/1)."""
    judge_prompt = (
        "You are a strict grader. Determine whether the [Model Final Answer] is semantically equivalent "
        "to the [Ground Truth Answer]. Output 1 if correct, 0 if incorrect. "
        "ignore any formatting differences"
        "Only output a single character: 0 or 1. No explanation.\n\n"
        f"[Ground Truth Answer]\n{ground_truth_answer}\n\n"
        f"[Model Final Answer]\n{final_output}\n"
    )

    try:
        resp = client.chat.completions.create(
            model=GRADER_MODEL_NAME,
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=5,
            temperature=0.0,
        )
        score_str = (resp.choices[0].message.content or "").strip()
        m = re.search(r"\b([01])\b", score_str)
        if m:
            return int(m.group(1))

        # Fallback: parse numeric-like output
        m2 = re.search(r"([0-9]*\.?[0-9]+)", score_str)
        if m2:
            return 1 if float(m2.group(1)) >= 0.5 else 0
    except Exception as e:
        print(f"⚠️ Final grading failed; fallback rule enabled: {type(e).__name__}: {e}")

    # Final fallback: heuristic string match
    gt = _normalize_answer_text(ground_truth_answer)
    pred = _normalize_answer_text(final_output)
    if not gt or not pred:
        return 0
    if gt == pred or gt in pred or pred in gt:
        return 1
    return 0


def score_node_by_judge_llm(client, node_input_text: str, node_output: str) -> float:
    """Scorer-2: when final answer is incorrect, grade node output against node input text (0~1)."""
    judge_prompt = (
        "You are a strict grader. Based on the [Task Input Text], evaluate how well the [Node Output] "
        "completes the task. Output only a decimal number between 0 and 1 (up to 2 decimals). "
        "No explanation.\n\n"
        f"[Task Input Text]\n{node_input_text}\n\n"
        f"[Node Output]\n{node_output}\n"
    )

    try:
        resp = client.chat.completions.create(
            model=GRADER_MODEL_NAME,
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=10,
            temperature=0.0,
        )
        score_str = (resp.choices[0].message.content or "").strip()
        match = re.search(r"([0-9]*\.?[0-9]+)", score_str)
        score = float(match.group(1)) if match else 0.0
    except Exception as e:
        print(f"⚠️ Node grading failed, default to 0: {type(e).__name__}: {e}")
        score = 0.0

    return float(max(0.0, min(1.0, score)))


def build_node_input_text(node: DAGNode) -> str:
    """Build model input text using current node task + predecessor concise answers only."""
    predecessor_qa = []
    for p in node.predecessors:
        concise_ans = p.answer_content if getattr(p, "answer_content", "") else _extract_answer_tag(p.output_content)
        predecessor_qa.append(
            f"Predecessor Node {p.node_id}\n"
            f"Question: {p.task_description}\n"
            f"Answer: {concise_ans}"
        )
    predecessor_qa_text = "\n\n".join(predecessor_qa) if predecessor_qa else "No predecessor nodes."

    input_text = (
        f"Current Node Task: {node.task_description}\n\n"
        f"Predecessor Q&A:\n{predecessor_qa_text}\n\n"
        "You may include reasoning, but you MUST provide the final concise answer wrapped in <answer>...</answer>. "
        "Only the content inside <answer> will be used by downstream nodes and grading."
    )
    return input_text


def execute_and_evaluate_dag(graph: DAGGraph, problem_desc: str, ground_truth_answer: str,
                             ucb_model, feature_extractor, client):
    """
    Execute DAG nodes, select models, grade outputs, and return training observations.
    - Model input text: current node task + predecessor concise answers
    - UCB vector source text: model input text only (subject removed)
    - Two-level scoring: final answer 0/1, then node-level 0~1
    """
    layers_dict = graph.compute_layers()
    sorted_layers = sorted(layers_dict.keys())

    execution_attempts = []

    # 1) Execute node by node
    for layer in sorted_layers:
        for node in layers_dict[layer]:
            model_input_text = build_node_input_text(node)
            node.input_context = model_input_text

            # Vector source text = model input text only (dataset has no subject)
            vector_source_text = model_input_text
            x_tn = feature_extractor.get_feature_vector(vector_source_text)
            node.feature_vector = x_tn

            chosen_model, scores_info = ucb_model.select_model(x_tn)
            node.selected_model = chosen_model

            llm_input_messages = [
                {"role": "user", "content": model_input_text}
            ]

            try:
                resp = client.chat.completions.create(
                    model=chosen_model,
                    messages=llm_input_messages,
                    max_tokens=768,
                    temperature=0.1,
                )
                node.output_content = (resp.choices[0].message.content or "").strip()
            except Exception as e:
                print(f"⚠️ Node {node.node_id} model call failed: {type(e).__name__}: {e}")
                node.output_content = ""

            node.answer_content = _extract_answer_tag(node.output_content)

            execution_attempts.append({
                "step": len(execution_attempts) + 1,
                "node_id": node.node_id,
                "predecessor_node_ids": [p.node_id for p in node.predecessors],
                "task_desc": node.task_description,
                "chosen_model": node.selected_model,
                "ucb_scores": scores_info,
                "output": node.output_content,
                "parsed_answer": node.answer_content,
                "llm_input_messages": llm_input_messages,
                "vector_source_text": vector_source_text,
                "reward": None,
            })

    # 2) Aggregate final output using concise extracted answers from terminal nodes
    terminal_nodes = [n for n in graph.nodes.values() if not n.successors]
    final_output = "\n".join([n.answer_content for n in terminal_nodes]).strip()

    # Scorer-1: final answer vs ground truth, 0/1 (using GRADER_MODEL_NAME)
    final_correct = score_final_answer_by_gt(client, ground_truth_answer, final_output)

    # 3) Node rewards
    observations = []
    if final_correct == 1:
        # All nodes reward=1
        for node in graph.nodes.values():
            node.reward = 1.0
            observations.append((node.feature_vector, node.selected_model, node.reward))
        for a in execution_attempts:
            a["reward"] = 1.0
        final_status = "success"
    else:
        # Scorer-2: per-node grading 0~1, use extracted concise answer
        for node in graph.nodes.values():
            node.reward = score_node_by_judge_llm(client, node.input_context, node.answer_content)
            observations.append((node.feature_vector, node.selected_model, node.reward))
        # Fill rewards back into attempts
        reward_map = {n.node_id: n.reward for n in graph.nodes.values()}
        for a in execution_attempts:
            a["reward"] = float(reward_map.get(a["node_id"], 0.0))
        final_status = "failure"

    return observations, final_output, final_status, execution_attempts, int(final_correct)

In [85]:
# TODO: 实现 GreedyNeuralUCBModel 类，包含参数初始化、更新、预测等方法。（抄过来还没修改完）
class GreedyNeuralUCBModel:
    def __init__(
        self,
        model_names,
        feature_dim=EMBEDDING_DIM,
        learning_rate=LEARNING_RATE,
        reg=REG,
        m=WIDTH,
        J=GD_STEPS,
        gamma=GEMMA,
        L=L,
        batch_size=BATCH_SIZE,
        seed=42,
    ):
        self.model_names = model_names
        self.feature_dim = feature_dim
        self.learning_rate = learning_rate
        self.reg = reg
        self.m = m
        self.J = J
        self.gamma = gamma
        self.batch_size = batch_size
        self.L = L

        self.model_name_to_index = {name: i for i, name in enumerate(model_names)}
        self.rng = np.random.default_rng(seed)

        self.buffer = [] # 经验回放缓冲区，用于暂存当前的观测数据 (x_vector, model_name, reward)
        self.t = 0       # 记录 query round t，即当前处理了多少个节点

        # 网络结构：输入 -> (L-1 个隐藏层, 宽度 m) -> 输出 1
        num_hidden = max(self.L - 1, 1)
        layer_sizes = [self.feature_dim] + [self.m] * num_hidden + [1]
        self.layer_sizes = layer_sizes

        def _block_diag(W):
            z = np.zeros_like(W)
            top = np.concatenate([W, z], axis=1)
            bottom = np.concatenate([z, W], axis=1)
            return np.concatenate([top, bottom], axis=0)

        def _init_params():
            params = []
            for li, (in_dim, out_dim) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:]), start=1):
                # 初始化满足 NTK 结构：
                # W_l = [[W, 0],[0, W]]，W ~ N(0, 4/m)  (l in [L-1])
                # W_L = [w^T, -w^T]，w ~ N(0, 2/m)
                if li < len(layer_sizes) - 1:
                    if in_dim == out_dim and in_dim % 2 == 0:
                        half = in_dim // 2
                        W = self.rng.normal(0.0, np.sqrt(4.0 / self.m), size=(half, half)).astype(np.float32)
                        w = _block_diag(W).astype(np.float32)
                    else:
                        w = self.rng.normal(0.0, np.sqrt(4.0 / self.m), size=(out_dim, in_dim)).astype(np.float32)
                    b = np.zeros(out_dim, dtype=np.float32)
                else:
                    if in_dim % 2 == 0:
                        half = in_dim // 2
                        w_vec = self.rng.normal(0.0, np.sqrt(2.0 / self.m), size=(half,)).astype(np.float32)
                        w = np.concatenate([w_vec, -w_vec], axis=0).reshape(1, -1).astype(np.float32)
                        if w.shape[1] != in_dim:
                            w = self.rng.normal(0.0, np.sqrt(2.0 / self.m), size=(out_dim, in_dim)).astype(np.float32)
                    else:
                        w = self.rng.normal(0.0, np.sqrt(2.0 / self.m), size=(out_dim, in_dim)).astype(np.float32)
                    b = np.zeros(out_dim, dtype=np.float32)
                params.append((w, b))
            return params

        def _param_count(params):
            total = 0
            for w, b in params:
                total += w.size + b.size
            return total

        def _build_net_from_params(params):
            net = LLMNet(input_dim=self.feature_dim, width=self.m, L=self.L)
            linear_layers = [m for m in net.net if isinstance(m, nn.Linear)]
            for (w_np, b_np), layer in zip(params, linear_layers):
                with torch.no_grad():
                    layer.weight.copy_(torch.tensor(w_np, dtype=layer.weight.dtype))
                    layer.bias.copy_(torch.tensor(b_np, dtype=layer.bias.dtype))
            return net

        def _flatten_params(net):
            return torch.cat([p.detach().reshape(-1) for p in net.parameters()]).cpu().numpy()

        self.models = {}
        # ==========================================
        # 算法 Algorithm 1 第 1 行: For each LLM j in [M], 初始化网络参数和协方差矩阵 Z
        # ==========================================
        for model_name in self.model_names:
            params = _init_params()
            p = _param_count(params)
            net = _build_net_from_params(params)
            net = net.to(DEVICE)  # 保证模型与输入位于同一设备
            theta0 = _flatten_params(net)

            self.models[model_name] = {
                "params": params,
                "net": net,
                "theta": np.copy(theta0),  # 当前轮次的网络参数
                "theta0": np.copy(theta0), # 初始网络参数（用于正则化约束，防止灾难性遗忘）
                # 算法要求 set Z_{0,j} = \lambda I。
                # 由于全矩阵太大且求逆太慢，这里使用【对角线近似】(Diagonal Approximation)
                # 即用一个长度为 p 的一维数组代替矩阵，初始值为正则化参数 λ (self.reg)
                "Z_diag": np.ones(p, dtype=np.float32) * self.reg,
                "last_call_time": 0,
            }

    def add_observations_and_update(self, observations):
        """
        对应算法图第 20-31 行的逻辑：收集真实反馈并更新神经网络参数。
        observations: 一个列表，元素格式为 (特征向量x, 选中的大模型名称, 真实得分reward)
        """
        self.t += 1

        # ==========================================
        # 算法 Algorithm 1 第 20 行: Add observation tuples to buffer B
        # ==========================================
        for obs in observations:
            self.buffer.append(obs)

        # ==========================================
        # 算法 Algorithm 1 第 21 行: if t mod B_batch == 0 then (是否达到了设定的批次大小)
        # ==========================================
        if self.t % self.batch_size == 0 and len(self.buffer) > 0:

            # 算法 Algorithm 1 第 22 行: for each LLM j in [M] do
            for model_name, model_state in self.models.items():

                # 算法 Algorithm 1 第 23 行: Let B_j be the subset of buffer B where LLM j was selected
                # 过滤出“当前这个大模型”被选中时产生的数据样本
                B_j = [obs for obs in self.buffer if obs[1] == model_name]
                if not B_j: # 如果这个模型在这个批次里一次都没被选过，就跳过不更新
                    continue

                net = model_state["net"].to(DEVICE)
                model_state["net"] = net
                net.train() # 开启训练模式
                optimizer = optim.Adam(net.parameters(), lr=self.learning_rate)

                # 将该模型专属的经验数据转换为 PyTorch 张量
                xs = torch.tensor(np.array([d[0] for d in B_j]), dtype=torch.float32).to(DEVICE)
                ys = torch.tensor(np.array([d[2] for d in B_j]), dtype=torch.float32).to(DEVICE)
                theta0 = torch.tensor(model_state["theta0"], dtype=torch.float32).to(DEVICE)

                # ==========================================
                # 算法 Algorithm 1 第 25 行: Update \theta_{t,j} 最小化 L2 loss (包含正则项)
                # ==========================================
                # 进行 J 次梯度下降迭代 (for J iterations)
                for _ in range(self.J):
                    optimizer.zero_grad()
                    preds = net(xs)

                    # MSE 损失: 1/2 * (预测值 - 真实得分)^2
                    mse = 0.5 * torch.mean((preds - ys) ** 2)

                    # NTK 理论要求的正则项: m * λ / 2 * ||\theta - \theta_0||^2
                    # 这个项可以防止网络参数偏离初始值太远，保证理论收敛性
                    theta = torch.cat([p.reshape(-1) for p in net.parameters()])
                    reg_term = 0.5 * self.m * self.reg * torch.sum((theta - theta0) ** 2)

                    # 总损失 = 预测误差 + 正则项
                    loss = mse + reg_term
                    loss.backward()
                    optimizer.step()

                # 保存更新后的网络参数
                model_state["theta"] = torch.cat([p.detach().reshape(-1) for p in net.parameters()]).cpu().numpy().astype(np.float32)

                # ==========================================
                # 算法 Algorithm 1 第 24 行: Update Z_{t,j} 协方差矩阵（用于后续计算探索奖励）
                # ==========================================
                for x_val in xs:
                    x_single = x_val.unsqueeze(0)
                    net.zero_grad(set_to_none=True)
                    # 前向传播求值
                    y = net(x_single).sum()

                    # 反向传播求【网络输出对输入参数的梯度】 g = \nabla f(x; \theta)
                    grads = torch.autograd.grad(y, net.parameters(), retain_graph=False, create_graph=False)
                    g = torch.cat([grad.reshape(-1) for grad in grads]).detach().cpu().numpy()

                    # 理论上是 Z = Z + g * g^T / m。由于我们用了对角线近似，这里变成了向量逐元素的平方累加
                    model_state["Z_diag"] += (g ** 2) / float(self.m)

            # ==========================================
            # 算法 Algorithm 1 第 27 行: Clear buffer B = ∅
            # ==========================================
            self.buffer = []

    def select_model(self, feature_vector):
        """
        对应算法图第 7-11 行的逻辑：遍历所有大模型，计算 UCB(上限置信区间) 分数，选出得分最高的模型。
        """
        best_model = None
        max_ucb = -float('inf')
        model_scores_info = {}

        # 构造节点特征向量 x_{t,n}
        x = torch.tensor(feature_vector, dtype=torch.float32).to(DEVICE)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        # 算法 Algorithm 1 第 7 行: for each LLM j in [M] do
        for model_name, model_state in self.models.items():
            net = model_state["net"].to(DEVICE)
            model_state["net"] = net
            Z_diag = model_state["Z_diag"]

            net.eval() # 开启评估模式，关闭 Dropout 等
            net.zero_grad(set_to_none=True)

            # ==========================================
            # 算法 Algorithm 1 第 8 行: Compute estimated score \hat{v}_{t,n,j}
            # ==========================================
            # 预估得分 = 神经网络的输出分数
            y_pred_tensor = net(x).sum()
            v_hat = y_pred_tensor.item()

            # 求预测值对网络参数的梯度 g = \nabla f(x; \theta)
            grads = torch.autograd.grad(y_pred_tensor, net.parameters(), retain_graph=False, create_graph=False)
            g = torch.cat([grad.reshape(-1) for grad in grads]).detach().cpu().numpy()

            # ==========================================
            # 算法 Algorithm 1 第 9 行: Compute UCB u_{t,n,j}
            # ==========================================
            # UCB = 预估得分 + 探索奖励(Bonus)
            # 原始公式里的矩阵运算 Z^{-1} 被简化为了向量点除 (Z_diag + 1e-8 防止除零)
            bonus = self.gamma * np.sqrt(np.sum((g ** 2) / (self.m * (Z_diag + 1e-8))))
            ucb_score = v_hat + bonus

            model_scores_info[model_name] = {
                "pred_score": round(v_hat, 4),
                "bonus": round(bonus, 4),
                "total_ucb": round(ucb_score, 4)
            }

            # ==========================================
            # 算法 Algorithm 1 第 11 行: Select LLM with max UCB
            # ==========================================
            if ucb_score > max_ucb:
                max_ucb = ucb_score
                best_model = model_name

        # 返回选出的最强模型名称，以及所有模型的打分细节
        return best_model, model_scores_info

    def save_model_state(self, file_path):
        """Save full UCB state (config + per-model NN params + statistics)."""
        out_dir = os.path.dirname(file_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)

        state = {
            "config": {
                "feature_dim": self.feature_dim,
                "learning_rate": self.learning_rate,
                "reg": self.reg,
                "m": self.m,
                "J": self.J,
                "gamma": self.gamma,
                "L": self.L,
                "batch_size": self.batch_size,
                "model_names": list(self.model_names),
            },
            "meta": {
                "t": int(self.t),
                "buffer_size": len(self.buffer),
            },
            "models": {},
        }

        for model_name, model_state in self.models.items():
            net = model_state["net"].to("cpu")
            nn_state = {k: v.detach().cpu() for k, v in net.state_dict().items()}
            state["models"][model_name] = {
                "net": nn_state,
                "theta": np.asarray(model_state["theta"], dtype=np.float32),
                "theta0": np.asarray(model_state["theta0"], dtype=np.float32),
                "Z_diag": np.asarray(model_state["Z_diag"], dtype=np.float32),
                "last_call_time": int(model_state.get("last_call_time", 0)),
            }
            # move back for consistency
            model_state["net"] = net.to(DEVICE)

        torch.save(state, file_path)
        print(f"✅ Model state saved to: {file_path}")

    def load_model_state(self, file_path):
        """Load full UCB state from file and restore per-model parameters/statistics."""
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"State file not found: {file_path}")

        try:
            state = torch.load(file_path, map_location="cpu", weights_only=False)
        except TypeError:
            state = torch.load(file_path, map_location="cpu")

        if not isinstance(state, dict) or "models" not in state:
            raise ValueError("Invalid state file format: missing 'models'.")

        loaded_models = state.get("models", {})
        loaded_meta = state.get("meta", {})

        for model_name, cur_state in self.models.items():
            if model_name not in loaded_models:
                continue

            m_state = loaded_models[model_name]

            # 1) load network weights
            net = cur_state["net"].to("cpu")
            net.load_state_dict(m_state["net"], strict=True)
            net = net.to(DEVICE)
            cur_state["net"] = net

            # 2) load theta/theta0
            theta = m_state.get("theta", None)
            theta0 = m_state.get("theta0", None)
            if theta is None:
                theta = torch.cat([p.detach().reshape(-1) for p in net.parameters()]).cpu().numpy()
            if theta0 is None:
                theta0 = np.copy(theta)
            cur_state["theta"] = np.asarray(theta, dtype=np.float32)
            cur_state["theta0"] = np.asarray(theta0, dtype=np.float32)

            # 3) load / validate Z_diag
            z_loaded = m_state.get("Z_diag", None)
            p_dim = cur_state["theta"].shape[0]
            if z_loaded is None:
                cur_state["Z_diag"] = np.ones(p_dim, dtype=np.float32) * self.reg
            else:
                z_arr = np.asarray(z_loaded, dtype=np.float32).reshape(-1)
                if z_arr.shape[0] != p_dim:
                    print(f"⚠️ Z_diag shape mismatch for {model_name}; reset to λI diagonal.")
                    z_arr = np.ones(p_dim, dtype=np.float32) * self.reg
                cur_state["Z_diag"] = z_arr

            # 4) misc
            cur_state["last_call_time"] = int(m_state.get("last_call_time", 0))

        self.t = int(loaded_meta.get("t", self.t))
        self.buffer = []
        print(f"✅ Model state loaded from: {file_path}")
        return True

In [ ]:
# # 在 fused_qa_500 中抽取 20 题，测试 Decompose_MODEL_NAME 的分解能力
# # 输入问题字段使用: problem

# from pathlib import Path


# def _find_fused_path() -> Path:
#     candidates = [
#         Path.cwd() / "data" / "fused_qa_500.json",
#         Path.cwd().parent / "data" / "fused_qa_500.json",
#     ]
#     for p in candidates:
#         if p.exists():
#             return p
#     raise FileNotFoundError("未找到 data/fused_qa_500.json")


# fused_path = _find_fused_path()
# with open(fused_path, "r", encoding="utf-8") as f:
#     fused_data = json.load(f)

# assert isinstance(fused_data, list), "fused_qa_500.json 应为 list 结构"
# assert "decompose_query_to_dag" in globals(), "请先运行定义 decompose_query_to_dag 的单元。"

# N_TEST = 20
# selected = []
# for i, rec in enumerate(fused_data):
#     q = rec.get("problem", "")
#     if isinstance(q, str) and q.strip():
#         pid = str(rec.get("problem_id", i))
#         selected.append((pid, q.strip(), rec.get("subject", "unknown")))
#     if len(selected) >= N_TEST:
#         break

# print(f"Using Decompose model: {Decompose_MODEL_NAME}")
# print(f"Dataset: {fused_path.name}")
# print(f"Selected: {len(selected)}\n")

# all_results = []

# for idx, (pid, question, subject) in enumerate(selected, start=1):
#     print("=" * 80)
#     print(f"[{idx}/{len(selected)}] problem_id={pid} | subject={subject}")
#     print(f"Question: {question[:220]}{'...' if len(question) > 220 else ''}")

#     try:
#         graph = decompose_query_to_dag(question, pid, client)
#         node_count = len(graph.nodes)
#         edge_count = sum(len(n.successors) for n in graph.nodes.values())

#         print(f"Decomposition: nodes={node_count}, edges={edge_count}")
#         print("--- Nodes ---")
#         for nid, node in graph.nodes.items():
#             pred_ids = [p.node_id for p in node.predecessors]
#             succ_ids = [s.node_id for s in node.successors]
#             print(f"- {nid}: preds={pred_ids}, succs={succ_ids}")
#             print(f"  task: {node.task_description}")

#         all_results.append({
#             "problem_id": pid,
#             "subject": subject,
#             "ok": True,
#             "nodes": node_count,
#             "edges": edge_count,
#         })
#     except Exception as e:
#         print(f"❌ Failed: {type(e).__name__}: {e}")
#         all_results.append({
#             "problem_id": pid,
#             "subject": subject,
#             "ok": False,
#             "nodes": 0,
#             "edges": 0,
#             "error": str(e),
#         })

# print("\n" + "=" * 80)
# ok_n = sum(int(r["ok"]) for r in all_results)
# avg_nodes = (sum(r["nodes"] for r in all_results if r["ok"]) / ok_n) if ok_n else 0.0
# print(f"Summary: success={ok_n}/{len(all_results)}, avg_nodes={avg_nodes:.2f}")

In [86]:
def main_training_loop(dataset: list, ucb_model, feature_extractor, client, recorder):
    """
    算法第 2-19 行主控流程（含执行、两级评分、参数更新）。
    """
    for idx, record in enumerate(tqdm(dataset, desc="Processing Queries")):
        raw_query = record.get("problem", record.get("question", record.get("query", "")))
        problem_id = record.get("problem_id", record.get("id", str(idx)))
        # subject = record.get("subject", "unknown")  # final_evaluation_dataset.json has no subject
        gt_answer = record.get("answer", record.get("answers", ""))

        if not raw_query:
            continue

        # 1) 动态分解
        graph = decompose_query_to_dag(raw_query, str(problem_id), client)
        problem_desc = getattr(graph, "problem_description", raw_query)

        # 2) 图执行与评估
        obs, final_out, final_status, attempts, final_correct = execute_and_evaluate_dag(
            graph=graph,
            problem_desc=problem_desc,
            # problem_subject=subject,  # removed: dataset has no subject
            ground_truth_answer=gt_answer,
            ucb_model=ucb_model,
            feature_extractor=feature_extractor,
            client=client,
        )

        # 3) 记录
        recorder.add_record(
            problem_id=str(problem_id),
            question=raw_query,
            expected_answers=gt_answer,
            final_status=final_status,
            attempts=attempts,
        )

        # 4) 更新 UCB 参数
        ucb_model.add_observations_and_update(obs)

        print(f"\n[problem_id={problem_id}] final_correct={final_correct}, final_status={final_status}")
        print(f"final_output: {final_out[:200]}{'...' if len(final_out) > 200 else ''}")

    print("🎉 所有查询处理并训练完毕！")


# ==========================================
# 🚀 启动执行模块
# ==========================================
# 1. 实例化核心组件
ucb_model = GreedyNeuralUCBModel(model_names=AVAILABLE_LLMS)
recorder = ResultRecorder("execution_records.json")

# 2. 读取数据集
my_dataset = load_dataset("../data/final_evaluation_dataset_500.json")

# # 3. 运行（默认先小样本 smoke test）
# main_training_loop(my_dataset[:2], ucb_model, extractor, client, recorder)
# main_training_loop(my_dataset, ucb_model, extractor, client, recorder)

✅ 成功加载数据集: ../data/final_evaluation_dataset_500.json


In [93]:
# 小批量测试：使用 5 个问题快速验证端到端流程
SMOKE_N = 5

# 若 extractor 尚未成功初始化，则在此重试一次
if "extractor" not in globals() or extractor is None:
    print("⚠️ extractor 不存在，正在尝试初始化...")
    extractor = FeatureExtractor()

# 准备组件（与主训练流程一致）
ucb_model = GreedyNeuralUCBModel(model_names=AVAILABLE_LLMS)
recorder = ResultRecorder("execution_records_smoke_5.json")

# 加载数据并截取前 5 条
dataset_path = "../data/final_evaluation_dataset.json"
my_dataset = load_dataset(dataset_path)
small_batch = my_dataset[:SMOKE_N] if isinstance(my_dataset, list) else []

print(f"🚀 Start smoke test with {len(small_batch)} samples")
main_training_loop(small_batch, ucb_model, extractor, client, recorder)
print("✅ Smoke test finished.")

✅ 成功加载数据集: /home/guisy/MyExperiment/LLM_DAG_Routing_02/data/final_evaluation_dataset_500.json
🚀 Start smoke test with 5 samples


Processing Queries:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Successfully decomposed query into a DAG with 1 nodes.


Processing Queries:  20%|██        | 1/5 [00:18<01:14, 18.62s/it]


[problem_id=1] final_correct=0, final_status=failure
final_output: The Sixth Sense
✅ Successfully decomposed query into a DAG with 3 nodes.


...


[problem_id=3] final_correct=1, final_status=success
final_output: J
✅ Successfully decomposed query into a DAG with 4 nodes.


Processing Queries:  80%|████████  | 4/5 [02:21<00:35, 35.77s/it]


[problem_id=4] final_correct=1, final_status=success
final_output: E
✅ Successfully decomposed query into a DAG with 2 nodes.


Processing Queries: 100%|██████████| 5/5 [02:34<00:00, 30.81s/it]


[problem_id=5] final_correct=1, final_status=success
final_output: Am Rong
🎉 所有查询处理并训练完毕！
✅ Smoke test finished.


In [87]:
from datetime import datetime
from pathlib import Path
import json


def _get_record_dir() -> Path:
    """优先使用已有 record 目录；若不存在则在当前工作目录创建。"""
    candidates = [Path.cwd() / "record", Path.cwd().parent / "record"]
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    p = Path.cwd() / "record"
    p.mkdir(parents=True, exist_ok=True)
    return p


def _resolve_record_path(name_or_path: str, record_dir: Path) -> str:
    p = Path(name_or_path)
    if p.is_absolute():
        p.parent.mkdir(parents=True, exist_ok=True)
        return str(p)
    return str(record_dir / p.name)


def train_with_controls(
    dataset_path: str = "../data/final_evaluation_dataset_500.json",
    train_size: int | None = None,
    use_previous_params: bool = True,
    state_file: str = "ucb_model_state_latest.pt",
    raw_record_file: str = "execution_records_controlled.json",
    per_question_report_file: str = "per_question_metrics.json",
):
    """
    可控训练入口：
    1) train_size: 人为控制训练题数（None=全量）
    2) use_previous_params: 是否加载历史参数
    3) 训练后自动保存参数
    4) 生成每题正确性与回答模型记录文件
    """
    if "extractor" not in globals() or extractor is None:
        print("⚠️ extractor 不存在，正在尝试初始化...")
        globals()["extractor"] = FeatureExtractor()

    record_dir = _get_record_dir()
    state_path = _resolve_record_path(state_file, record_dir)
    raw_record_path_name = Path(raw_record_file).name
    report_path = _resolve_record_path(per_question_report_file, record_dir)

    dataset = load_dataset(dataset_path)
    if not isinstance(dataset, list) or len(dataset) == 0:
        raise ValueError("数据集为空或格式不正确，无法训练。")

    if train_size is not None:
        train_size = max(0, int(train_size))
        dataset = dataset[:train_size]

    if len(dataset) == 0:
        raise ValueError("train_size 截断后数据为空，请调整 train_size。")

    ucb_model_local = GreedyNeuralUCBModel(model_names=AVAILABLE_LLMS)

    if use_previous_params and Path(state_path).exists():
        try:
            ucb_model_local.load_model_state(state_path)
            print(f"🔁 已加载历史参数: {state_path}")
        except Exception as e:
            print(f"⚠️ 历史参数加载失败，将从头训练: {type(e).__name__}: {e}")
    else:
        print("🆕 从随机初始化参数开始训练。")

    recorder_local = ResultRecorder(raw_record_path_name)

    per_question_metrics = []

    for idx, record in enumerate(tqdm(dataset, desc="Controlled Training")):
        raw_query = record.get("problem", record.get("question", record.get("query", "")))
        problem_id = str(record.get("problem_id", record.get("id", idx)))
        gt_answer = record.get("answer", record.get("answers", ""))

        if not raw_query:
            continue

        graph = decompose_query_to_dag(raw_query, problem_id, client)
        problem_desc = getattr(graph, "problem_description", raw_query)

        obs, final_out, final_status, attempts, final_correct = execute_and_evaluate_dag(
            graph=graph,
            problem_desc=problem_desc,
            ground_truth_answer=gt_answer,
            ucb_model=ucb_model_local,
            feature_extractor=extractor,
            client=client,
        )

        recorder_local.add_record(
            problem_id=problem_id,
            question=raw_query,
            expected_answers=gt_answer,
            final_status=final_status,
            attempts=attempts,
        )

        ucb_model_local.add_observations_and_update(obs)

        terminal_node_ids = [n.node_id for n in graph.nodes.values() if not n.successors]
        attempt_map = {str(a.get("node_id")): a for a in attempts}
        answer_models = [attempt_map[nid].get("chosen_model") for nid in terminal_node_ids if nid in attempt_map]
        used_models = sorted({a.get("chosen_model") for a in attempts if a.get("chosen_model")})

        per_question_metrics.append({
            "problem_id": problem_id,
            "is_correct": int(final_correct),
            "final_status": final_status,
            "answer_model": answer_models[0] if len(answer_models) == 1 else answer_models,
            "used_models": used_models,
            "steps_taken": len(attempts),
            "expected_answer": gt_answer,
            "final_output": final_out,
        })

    # 自动保存训练后的参数
    ucb_model_local.save_model_state(state_path)

    total = len(per_question_metrics)
    correct = sum(x["is_correct"] for x in per_question_metrics)
    accuracy = (correct / total) if total else 0.0

    report_payload = {
        "config": {
            "dataset_path": dataset_path,
            "train_size": train_size if train_size is not None else "all",
            "use_previous_params": bool(use_previous_params),
            "state_file": state_path,
            "raw_record_file": str(_get_record_dir() / raw_record_path_name),
            "generated_at": datetime.now().isoformat(timespec="seconds"),
        },
        "summary": {
            "total_questions": total,
            "correct_questions": correct,
            "accuracy": round(accuracy, 4),
        },
        "per_question": per_question_metrics,
    }

    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report_payload, f, ensure_ascii=False, indent=2)

    print("\n✅ 训练完成")
    print(f"- 准确率: {correct}/{total} = {accuracy:.2%}")
    print(f"- 参数文件: {state_path}")
    print(f"- 逐题报告: {report_path}")
    print(f"- 详细执行记录: {str(_get_record_dir() / raw_record_path_name)}")

    # 便于后续继续使用
    globals()["ucb_model"] = ucb_model_local

    return report_payload


#===== 使用示例（按需修改） =====
report = train_with_controls(
    dataset_path="../data/final_evaluation_dataset_500.json",
    train_size=200,                 # 可人为控制训练题数
    use_previous_params=False,      # True=加载历史参数；False=从头训练
    state_file="ucb_model_state_latest.pt",
    raw_record_file="execution_records_train_50.json",
    per_question_report_file="per_question_metrics_train_50.json",
)
report["summary"]

✅ 成功加载数据集: ../data/final_evaluation_dataset_500.json
🆕 从随机初始化参数开始训练。


Controlled Training:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Successfully decomposed query into a DAG with 2 nodes.


Controlled Training:   0%|          | 1/200 [00:21<1:11:59, 21.70s/it]

✅ Successfully decomposed query into a DAG with 3 nodes.


...

Controlled Training: 100%|█████████▉| 199/200 [4:25:31<01:40, 100.19s/it]

✅ Successfully decomposed query into a DAG with 6 nodes.


Controlled Training: 100%|██████████| 200/200 [4:27:45<00:00, 80.33s/it] 

✅ Model state saved to: /home/guisy/MyExperiment/LLM_DAG_Routing_02/record/ucb_model_state_latest.pt

✅ 训练完成
- 准确率: 83/200 = 41.50%
- 参数文件: /home/guisy/MyExperiment/LLM_DAG_Routing_02/record/ucb_model_state_latest.pt
- 逐题报告: /home/guisy/MyExperiment/LLM_DAG_Routing_02/record/per_question_metrics_train_50.json
- 详细执行记录: /home/guisy/MyExperiment/LLM_DAG_Routing_02/record/execution_records_train_50.json


{'total_questions': 200, 'correct_questions': 83, 'accuracy': 0.415}